In [ ]:
# Import libraries. You may or may not use all of these.
!pip install -q git+https://github.com/tensorflow/docs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
  # %tensorflow_version only exists in Colab.
  %tensorflow_version 2.x
except Exception:
  pass
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

import tensorflow_docs as tfdocs
import tensorflow_docs.plots
import tensorflow_docs.modeling

In [ ]:
# Import data
!wget https://cdn.freecodecamp.org/project-data/health-costs/insurance.csv
dataset = pd.read_csv('insurance.csv')
dataset.tail()

In [ ]:
# Convert categorical features to numeric values.
dataset = pd.get_dummies(
    dataset,
    columns=['sex', 'smoker', 'region'],
    dtype='float32'
)

# Reproducible 80/20 train-test split.
train_dataset = dataset.sample(frac=0.8, random_state=42)
test_dataset = dataset.drop(train_dataset.index)

train_dataset = train_dataset.reset_index(drop=True)
test_dataset = test_dataset.reset_index(drop=True)

# Separate the target column exactly as required by the challenge.
train_labels = train_dataset.pop('expenses').astype('float32')
test_labels = test_dataset.pop('expenses').astype('float32')

train_dataset = train_dataset.astype('float32')
test_dataset = test_dataset.astype('float32')

# Normalize input features using statistics learned only from the training set.
normalizer = layers.Normalization(axis=-1)
normalizer.adapt(np.asarray(train_dataset, dtype=np.float32))

tf.keras.utils.set_random_seed(42)

model = keras.Sequential([
    layers.Input(shape=(train_dataset.shape[1],)),
    normalizer,
    layers.Dense(128, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae', 'mse']
)

early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_mae',
    patience=60,
    mode='min',
    restore_best_weights=True
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_mae',
    factor=0.5,
    patience=20,
    min_lr=1e-5,
    mode='min'
)

history = model.fit(
    train_dataset,
    train_labels,
    epochs=1000,
    validation_split=0.2,
    verbose=0,
    callbacks=[early_stopping, reduce_lr]
)

model.summary()


In [ ]:
# RUN THIS CELL TO TEST YOUR MODEL. DO NOT MODIFY CONTENTS.
# Test model by checking how well the model generalizes using the test set.
loss, mae, mse = model.evaluate(test_dataset, test_labels, verbose=2)

print("Testing set Mean Abs Error: {:5.2f} expenses".format(mae))

if mae < 3500:
  print("You passed the challenge. Great job!")
else:
  print("The Mean Abs Error must be less than 3500. Keep trying.")

# Plot predictions.
test_predictions = model.predict(test_dataset).flatten()

a = plt.axes(aspect='equal')
plt.scatter(test_labels, test_predictions)
plt.xlabel('True values (expenses)')
plt.ylabel('Predictions (expenses)')
lims = [0, 50000]
plt.xlim(lims)
plt.ylim(lims)
_ = plt.plot(lims,lims)
